In [2]:
!pip install earthengine-api geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.8 MB/s eta 0:00:00a 0:00:01


In [3]:
#This is to publish comment
import ee
ee.Authenticate()
ee.Initialize(project='osmgee')

In [4]:
knp_boundary = ee.FeatureCollection('projects/osmgee/assets/KNP')
sites = ee.FeatureCollection('projects/osmgee/assets/SitesLocation')
print('KNP features:', knp_boundary.size().getInfo())
print('Sites features:', sites.size().getInfo())

KNP features: 1
Sites features: 40


In [5]:
# Print the property names (field names) of the first feature
first_feature = sites.first()
print('Available fields:', first_feature.propertyNames().getInfo())

Available fields: ['XCoord', 'ClusterID', 'OBJECTID_1', 'FREQUENCY', 'ClusterID_', 'MEAN_PAR_L', 'Llllllatit', 'MEAN_GAP_F', 'system:index', 'YCoord', 'longitude']


In [6]:
# Pull PAR LAI, coordinates, and cluster ID into a client-side list to inspect
sites_info = sites.select(['ClusterID', 'MEAN_PAR_L', 'XCoord', 'YCoord', 'longitude', 'Llllllatit']) \
                   .getInfo()

# Print each feature's properties
for f in sites_info['features']:
    print(f['properties'])

{'ClusterID': '20', 'Llllllatit': 91.94278, 'MEAN_PAR_L': 1.6706171335, 'XCoord': 91.94278, 'YCoord': 24.95203, 'longitude': 24.95203}
{'ClusterID': '2', 'Llllllatit': 91.937182, 'MEAN_PAR_L': 3.0386616708, 'XCoord': 91.937182, 'YCoord': 24.952476, 'longitude': 24.952476}
{'ClusterID': '27', 'Llllllatit': 91.943748, 'MEAN_PAR_L': 2.1545768976, 'XCoord': 91.943748, 'YCoord': 24.94572, 'longitude': 24.94572}
{'ClusterID': '40', 'Llllllatit': 91.961718, 'MEAN_PAR_L': 0.285474385, 'XCoord': 91.961718, 'YCoord': 24.971386, 'longitude': 24.971386}
{'ClusterID': '11', 'Llllllatit': 91.93869, 'MEAN_PAR_L': 3.9239733615, 'XCoord': 91.93869, 'YCoord': 24.9520733333, 'longitude': 24.952073333}
{'ClusterID': '13', 'Llllllatit': 91.938551667, 'MEAN_PAR_L': 3.14804422833, 'XCoord': 91.9385516667, 'YCoord': 24.95474, 'longitude': 24.95474}
{'ClusterID': '25', 'Llllllatit': 91.94183, 'MEAN_PAR_L': 4.92397471843, 'XCoord': 91.94183, 'YCoord': 24.94815, 'longitude': 24.94815}
{'ClusterID': '32', 'Llllll

In [7]:
# Rebuild a clean FeatureCollection with unambiguous field names
def clean_feature(f):
    lon = ee.Number(f.get('XCoord'))   # confirmed: longitude
    lat = ee.Number(f.get('YCoord'))   # confirmed: latitude
    par_lai = ee.Number(f.get('MEAN_PAR_L'))
    cluster_id = f.get('ClusterID')
    
    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'cluster_id': cluster_id,
            'par_lai': par_lai,
            'longitude': lon,
            'latitude': lat
        }
    )

sites_clean = sites.map(clean_feature)

# Quick check
print('Cleaned feature count:', sites_clean.size().getInfo())
print('Sample feature:', sites_clean.first().getInfo())

Cleaned feature count: 40
Sample feature: {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [91.94278, 24.95203]}, 'id': '0000000000000000000c', 'properties': {'cluster_id': '20', 'latitude': 24.95203, 'longitude': 91.94278, 'par_lai': 1.6706171335}}


In [14]:
# Define field campaign center date and search window
field_start = ee.Date('2021-01-01')  # ~18 days before campaign start
field_end = ee.Date('2021-02-28')    # ~18 days after campaign end

# Load Sentinel-2 Surface Reflectance, filtered to KNP boundary and date window
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(knp_boundary) \
    .filterDate(field_start, field_end) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

# Check how many images are available
print('Number of available Sentinel-2 scenes:', s2_collection.size().getInfo())

# List their dates and cloud cover
image_list = s2_collection.toList(s2_collection.size())
n = s2_collection.size().getInfo()
for i in range(n):
    img = ee.Image(image_list.get(i))
    date = img.date().format('YYYY-MM-dd').getInfo()
    cloud = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f'{date}  —  cloud cover: {cloud:.1f}%')

Number of available Sentinel-2 scenes: 8
2021-01-01  —  cloud cover: 0.0%
2021-01-06  —  cloud cover: 0.0%
2021-01-16  —  cloud cover: 1.4%
2021-01-26  —  cloud cover: 1.6%
2021-01-31  —  cloud cover: 28.5%
2021-02-05  —  cloud cover: 0.0%
2021-02-10  —  cloud cover: 1.5%
2021-02-15  —  cloud cover: 0.0%


In [16]:
# Cloud masking function using the SCL (Scene Classification Layer) band
def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Keep only: vegetation(4), bare soil(5), water(6), unclassified(7)
    mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return image.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())

# Apply cloud masking to the collection
s2_masked = s2_collection.map(mask_s2_clouds)

# Build the median composite, clipped to KNP boundary
s2_composite = s2_masked.median().clip(knp_boundary)

# Compute spectral indices and add them to the composite (keeping original bands too)
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('EVI')
    
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    
    msavi = image.expression(
        '(2 * NIR + 1 - sqrt((2 * NIR + 1)**2 - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('MSAVI')
    
    return image.addBands([ndvi, gndvi, evi, savi, msavi])

s2_with_indices = add_indices(s2_composite)

# Select the raw bands we care about (10m/20m visible + NIR + red-edge, skip coarse atmospheric bands)
raw_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
index_bands = ['NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']

# Final image: raw bands + indices together
final_image = s2_with_indices.select(raw_bands + index_bands)

print('Bands in final image:', final_image.bandNames().getInfo())

Bands in final image: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']


In [17]:
# Function to create a 15m x 15m square buffer around each point
def make_square_plot(feature):
    point = feature.geometry()
    # buffer(7.5, ...) with square cap creates a 15m x 15m square centered on the point
    # Using bounds() on a small circular buffer approximates a square; 
    # for an exact square we build it directly from the point's coordinates
    half_side = 7.5  # half of 15m
    square = point.buffer(half_side, 1).bounds()  # bounds() of a buffer approximates square extent
    return feature.setGeometry(square)

# Apply to all 40 plots
sites_squares = sites_clean.map(make_square_plot)

# Quick check on one plot's geometry
sample_geom = sites_squares.first().geometry().getInfo()
print('Sample square plot geometry:', sample_geom)

Sample square plot geometry: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[91.94270594703491, 24.951963907026418], [91.94285098397934, 24.951963907026418], [91.94285098397934, 24.952097491494666], [91.94270594703491, 24.952097491494666], [91.94270594703491, 24.951963907026418]]]}


In [18]:
# Reduce the final_image (15 bands) over each 15x15m square plot, taking the mean
extracted = final_image.reduceRegions(
    collection=sites_squares,
    reducer=ee.Reducer.mean(),
    scale=10  # native resolution of the finest bands; reproject handled internally for 20m bands
)

# Pull results down to inspect
extracted_info = extracted.getInfo()

# Print first few features to check
for f in extracted_info['features'][:5]:
    print(f['properties'])
    print('---')

{'B11': 0.1390152943810237, 'B12': 0.06810412573855358, 'B2': 0.027017010599886843, 'B3': 0.044574914826560265, 'B4': 0.02836984464469849, 'B5': 0.07540498054314317, 'B6': 0.19427285351089596, 'B7': 0.23146959422380242, 'B8': 0.2360773164126062, 'B8A': 0.26436992752593, 'EVI': 0.431060697381481, 'GNDVI': 0.6827176737107361, 'MSAVI': 0.38023024131612426, 'NDVI': 0.7854034132914548, 'SAVI': 0.4070669318426147, 'cluster_id': '20', 'latitude': 24.95203, 'longitude': 91.94278, 'par_lai': 1.6706171335}
---
{'B11': 0.1033602367845005, 'B12': 0.05056245787126417, 'B2': 0.02682158691052081, 'B3': 0.04347491481757815, 'B4': 0.025658873774416415, 'B5': 0.05690238749411327, 'B6': 0.17125972314601062, 'B7': 0.21746825912503254, 'B8': 0.2577517078649062, 'B8A': 0.23607492088356116, 'EVI': 0.4746284376017571, 'GNDVI': 0.7081395854532604, 'MSAVI': 0.42207866086926743, 'NDVI': 0.8156945436901277, 'SAVI': 0.43845841667453944, 'cluster_id': '2', 'latitude': 24.952476, 'longitude': 91.937182, 'par_lai': 3

In [19]:
import pandas as pd

# Convert extracted GEE features into a clean pandas DataFrame
rows = []
for f in extracted_info['features']:
    rows.append(f['properties'])

df_extracted = pd.DataFrame(rows)

# Reorder columns for readability: identifiers first, then LAI, then bands/indices
id_cols = ['cluster_id', 'par_lai', 'longitude', 'latitude']
other_cols = [c for c in df_extracted.columns if c not in id_cols]
df_extracted = df_extracted[id_cols + other_cols]

# Display the full table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_extracted

,cluster_id,par_lai,longitude,latitude,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,EVI,GNDVI,MSAVI,NDVI,SAVI
0,20,1.670617,91.942780,24.952030,0.139015,0.068104,0.027017,0.044575,0.028370,0.075405,0.194273,0.231470,0.236077,0.264370,0.431061,0.682718,0.380230,0.785403,0.407067
1,2,3.038662,91.937182,24.952476,0.103360,0.050562,0.026822,0.043475,0.025659,0.056902,0.171260,0.217468,0.257752,0.236075,0.474628,0.708140,0.422079,0.815695,0.438458
2,27,2.154577,91.943748,24.945720,0.182999,0.105075,0.028932,0.049659,0.037179,0.089550,0.188746,0.228107,0.261835,0.252155,0.443024,0.681208,0.400174,0.751791,0.420949
3,40,0.285474,91.961718,24.971386,0.177583,0.110153,0.024830,0.039200,0.032634,0.090064,0.178097,0.212589,0.211046,0.235883,0.364982,0.687019,0.324918,0.733321,0.359329
4,11,3.923973,91.938690,24.952073,0.111651,0.053022,0.024702,0.040817,0.024098,0.065382,0.184818,0.229933,0.248737,0.245834,0.463579,0.716574,0.414344,0.821710,0.434497
5,13,3.148044,91.938552,24.954740,0.124945,0.059254,0.026353,0.040358,0.025978,0.072019,0.198838,0.244700,0.243550,0.269105,0.451856,0.714745,0.399931,0.805840,0.422957
6,25,4.923975,91.941830,24.948150,0.093438,0.051330,0.030898,0.040160,0.029426,0.053300,0.108200,0.144159,0.154748,0.151499,0.283431,0.582493,0.232176,0.675132,0.272502
7,32,1.538075,91.958823,24.968853,0.104768,0.051804,0.027164,0.037033,0.027657,0.057971,0.142010,0.178417,0.197944,0.209074,0.366475,0.684095,0.314937,0.753390,0.351325
8,33,3.215866,91.959881,24.966343,0.095074,0.046190,0.016397,0.025512,0.016486,0.049992,0.154161,0.193457,0.189613,0.216166,0.371173,0.763513,0.329812,0.840392,0.367394
9,36,3.479967,91.960536,24.968643,0.115038,0.058319,0.022761,0.035481,0.022560,0.059725,0.158080,0.194255,0.191448,0.213502,0.362928,0.684198,0.315586,0.785691,0.351403


In [21]:
from sklearn.model_selection import train_test_split
import numpy as np

# Define feature columns (10 raw bands + 5 indices) and target
feature_cols = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
                 'NDVI', 'EVI', 'SAVI', 'GNDVI', 'MSAVI']
target_col = 'par_lai'

X = df_extracted[feature_cols]
y = df_extracted[target_col]

# 80/20 split -> 32 train, 8 test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train PAR LAI range:', round(y_train.min(), 2), '-', round(y_train.max(), 2))
print('Test PAR LAI range:', round(y_test.min(), 2), '-', round(y_test.max(), 2))
print('Train PAR LAI mean:', round(y_train.mean(), 2))
print('Test PAR LAI mean:', round(y_test.mean(), 2))

Train PAR LAI range: 0.29 - 4.92
Test PAR LAI range: 0.86 - 5.41
Train PAR LAI mean: 2.34
Test PAR LAI mean: 2.83


In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_squared_error
import itertools

# Define hyperparameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5, 1.0]
}

# Generate all combinations
keys = list(param_grid.keys())
combinations = list(itertools.product(*param_grid.values()))
print(f'Total hyperparameter combinations to test: {len(combinations)}')

loo = LeaveOneOut()
results = []

for combo in combinations:
    params = dict(zip(keys, combo))
    rf = RandomForestRegressor(random_state=42, **params)
    
    # LOOCV: score = negative MSE, averaged across all 32 leave-one-out folds
    scores = cross_val_score(rf, X_train, y_train, cv=loo, scoring='neg_mean_squared_error')
    mean_mse = -scores.mean()
    rmse = np.sqrt(mean_mse)
    
    results.append({**params, 'loocv_rmse': rmse})

results_df = pd.DataFrame(results).sort_values('loocv_rmse').reset_index(drop=True)
print('Top 5 hyperparameter combinations by LOOCV RMSE:')
results_df.head(5)

Total hyperparameter combinations to test: 108
Top 5 hyperparameter combinations by LOOCV RMSE:


,n_estimators,max_depth,min_samples_leaf,max_features,loocv_rmse
0,100,NaN,4,1.0,0.877542
1,100,5.0,4,1.0,0.877542
2,100,10.0,4,1.0,0.877542
3,100,15.0,4,1.0,0.877542
4,300,10.0,4,1.0,0.889312


In [25]:
# Extract best hyperparameters from the LOOCV search
best_params = results_df.iloc[0][keys].to_dict()

# Clean up types (grid search combos can come back as numpy types from the DataFrame)
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['min_samples_leaf'] = int(best_params['min_samples_leaf'])
# max_depth can be None or an int
best_params['max_depth'] = None if pd.isna(best_params['max_depth']) else int(best_params['max_depth'])
# max_features can be 'sqrt' (string) or a float
if best_params['max_features'] not in ['sqrt', 'log2']:
    best_params['max_features'] = float(best_params['max_features'])

print('Best hyperparameters selected via LOOCV:')
print(best_params)

# Refit final RF model on all 32 training plots using best hyperparameters
final_rf = RandomForestRegressor(random_state=42, **best_params)
final_rf.fit(X_train, y_train)

# Evaluate once on the untouched 8-plot test set
from sklearn.metrics import r2_score, mean_absolute_error

y_pred_test = final_rf.predict(X_test)

r2 = r2_score(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
rrmse = (rmse / y_test.mean()) * 100

print()
print('=== Held-out test set performance (8 plots) ===')
print(f'R²:     {r2:.3f}')
print(f'RMSE:   {rmse:.3f}')
print(f'MAE:    {mae:.3f}')
print(f'rRMSE:  {rrmse:.1f}%')

Best hyperparameters selected via LOOCV:
{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 4, 'max_features': 1.0}

=== Held-out test set performance (8 plots) ===
R²:     -0.013
RMSE:   1.647
MAE:    1.324
rRMSE:  58.1%
